# Fanoron-telo AI: Dataset Generation & MLPClassifier Training

This notebook demonstrates how to generate a self-play dataset using Minimax/Alpha-Beta search and train a Multi-Layer Perceptron (Neural Network) classifier using Scikit-Learn. The trained model is saved as a Joblib file for production use in our FastAPI backend.

### Step 1: Install Dependencies
Uncomment and run if you need to install packages in your local environment.

In [ ]:
# !pip install numpy pandas scikit-learn joblib

### Step 2: Define Game Logic
We define the 3x3 Fanoron-telo board adjacency rules, winning lines, and the game loop state controller.

In [ ]:
import os
import json
import random
import time
import copy
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
import joblib

# Board connections (diagonals connect corner intersections to center)
ADJACENCY = {
    0: [1, 3, 4],
    1: [0, 2, 4],
    2: [1, 5, 4],
    3: [0, 6, 4],
    4: [0, 1, 2, 3, 5, 6, 7, 8],
    5: [2, 8, 4],
    6: [3, 7, 4],
    7: [6, 8, 4],
    8: [5, 7, 4]
}

# Winning patterns as bitmasks
WIN_PATTERNS = [7, 56, 448, 73, 146, 292, 273, 84]
WINNING_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

class FanoronGame:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [0] * 9
        self.current_player = 1
        self.winner = None
        self.phase = "placement"

    @property
    def bitmask_p1(self) -> int:
        mask = 0
        for i, val in enumerate(self.board):
            if val == 1: mask |= (1 << i)
        return mask

    @property
    def bitmask_p2(self) -> int:
        mask = 0
        for i, val in enumerate(self.board):
            if val == 2: mask |= (1 << i)
        return mask

    def check_win(self, player: int) -> bool:
        mask = self.bitmask_p1 if player == 1 else self.bitmask_p2
        for pattern in WIN_PATTERNS:
            if (mask & pattern) == pattern:
                return True
        return False

    def get_legal_moves(self) -> List[Dict]:
        if self.winner is not None:
            return []
        moves = []
        if self.phase == "placement":
            for c in range(9):
                if self.board[c] == 0:
                    moves.append({"type": "placement", "cell": c})
        else:
            for c in range(9):
                if self.board[c] == self.current_player:
                    for adj in ADJACENCY[c]:
                        if self.board[adj] == 0:
                            moves.append({"type": "movement", "from_cell": c, "to_cell": adj})
        return moves

    def make_move(self, move: Dict):
        if move["type"] == "placement":
            self.board[move["cell"]] = self.current_player
        else:
            self.board[move["from_cell"]] = 0
            self.board[move["to_cell"]] = self.current_player

        if self.check_win(self.current_player):
            self.winner = self.current_player
        else:
            if self.phase == "placement" and sum(1 for x in self.board if x != 0) >= 6:
                self.phase = "movement"
            self.current_player = 3 - self.current_player
            if len(self.get_legal_moves()) == 0:
                self.winner = 3 - self.current_player

### Step 3: Heuristic Evaluation & Alpha-Beta Search
We implement the heuristic board scoring and Alpha-Beta minimax algorithm used by the self-play solver.

In [ ]:
def evaluate_board(board: List[int], player: int) -> int:
    opponent = 3 - player
    mask_player = sum(1 << i for i, val in enumerate(board) if val == player)
    mask_opp = sum(1 << i for i, val in enumerate(board) if val == opponent)
    
    for pattern in WIN_PATTERNS:
        if (mask_player & pattern) == pattern: return 1000
        if (mask_opp & pattern) == pattern: return -1000

    score = 0
    for line in WINNING_LINES:
        p_c = sum(1 for c in line if board[c] == player)
        o_c = sum(1 for c in line if board[c] == opponent)
        emp = sum(1 for c in line if board[c] == 0)
        if p_c == 2 and emp == 1: score += 50
        elif o_c == 2 and emp == 1: score -= 50
        
    if board[4] == player: score += 10
    elif board[4] == opponent: score -= 10
    return score

def alphabeta_search(game: FanoronGame, depth: int, alpha: float, beta: float, is_max: bool) -> Tuple[Optional[Dict], float]:
    if game.winner is not None or depth == 0:
        return None, evaluate_board(game.board, 1 if is_max else 2)
        
    legal = game.get_legal_moves()
    if not legal:
        return None, evaluate_board(game.board, 1 if is_max else 2)
        
    best_move = None
    if is_max:
        val = -float('inf')
        for m in legal:
            game_copy = copy.deepcopy(game)
            game_copy.make_move(m)
            _, score = alphabeta_search(game_copy, depth - 1, alpha, beta, False)
            if score > val:
                val = score
                best_move = m
            alpha = max(alpha, val)
            if alpha >= beta: break
        return best_move, val
    else:
        val = float('inf')
        for m in legal:
            game_copy = copy.deepcopy(game)
            game_copy.make_move(m)
            _, score = alphabeta_search(game_copy, depth - 1, alpha, beta, True)
            if score < val:
                val = score
                best_move = m
            beta = min(beta, val)
            if alpha >= beta: break
        return best_move, val

### Step 4: Self-Play Simulation
We generate 10,000 games of self-play and store the results in `data/dataset.csv` and `data/dataset.json`.

In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

num_games = 10000
dataset = []
global_cache = {}

def get_optimal_move(game: FanoronGame) -> Dict:
    board_str = "".join(map(str, game.board))
    key = f"{board_str}-{game.current_player}-{game.phase}"
    if key in global_cache:
        return global_cache[key]
    
    # Perform depth-6 search
    best_move, _ = alphabeta_search(game, 6, -float('inf'), float('inf'), game.current_player == 1)
    if best_move is None:
        best_move = game.get_legal_moves()[0]
    global_cache[key] = best_move
    return best_move

print(f"Generating {num_games} self-play games...")
start_time = time.time()

for g_idx in range(num_games):
    game = FanoronGame()
    states = []
    turns = 0
    
    while game.winner is None and turns < 40:
        legal = game.get_legal_moves()
        if not legal: break
        
        opt_m = get_optimal_move(game)
        # 15% random exploration to discover diverse states
        chosen = random.choice(legal) if random.random() < 0.15 else opt_m
        
        best_cell = opt_m["cell"] if opt_m["type"] == "placement" else opt_m["to_cell"]
        legal_cells = [m["cell"] if m["type"] == "placement" else m["to_cell"] for m in legal]
        
        states.append((game.current_player, {
            "board": list(game.board),
            "current_player": game.current_player,
            "legal_moves": list(set(legal_cells)),
            "best_move": best_cell,
            "reward": 0
        }))
        game.make_move(chosen)
        turns += 1
        
    winner = game.winner if game.winner is not None else -1
    for pl, rec in states:
        rec["reward"] = 1 if winner == pl else (-1 if winner == (3 - pl) else 0)
        dataset.append(rec)
        
    if (g_idx + 1) % 2000 == 0:
        print(f"Simulated {g_idx+1} games...")

print(f"Done! Generated {len(dataset)} states in {time.time() - start_time:.1f}s")

# Save as JSON
with open("data/dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
    
# Save as CSV
rows = []
for r in dataset:
    row = {f"board_{i}": r["board"][i] for i in range(9)}
    row["current_player"] = r["current_player"]
    row["best_move"] = r["best_move"]
    row["reward"] = r["reward"]
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv("data/dataset.csv", index=False)
print("Saved data/dataset.csv and data/dataset.json")

### Step 5: Normalize and Train Model
We load `dataset.csv`, normalize the board features, instantiate a Multi-Layer Perceptron neural network, and train it to fit our self-play moves.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

df = pd.read_csv("data/dataset.csv")
X = []
y = []

def normalize(board, current_player):
    opp = 3 - current_player
    return [1.0 if v == current_player else (-1.0 if v == opp else 0.0) for v in board]

for _, row in df.iterrows():
    board = [int(row[f"board_{i}"]) for i in range(9)]
    pl = int(row["current_player"])
    bm = int(row["best_move"])
    X.append(normalize(board, pl))
    y.append(bm)

X = np.array(X)
y = np.array(y)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)

print("Training MLPClassifier Neural Network...")
clf = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42
)

start_t = time.time()
clf.fit(X_train, y_train)

acc_train = clf.score(X_train, y_train)
acc_val = clf.score(X_val, y_val)

print(f"Training finished in {time.time() - start_t:.1f}s")
print(f"Train Accuracy: {acc_train*100:.1f}%")
print(f"Val Accuracy: {acc_val*100:.1f}%")

# Save model weights
joblib.dump(clf, "models/fanoron.joblib")
print("Saved models/fanoron.joblib!")